[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saha23s/CD-MSC/blob/aaron%2Fpreprocessing/colab_dann.ipynb)

# CD-MSC — Supervised Contrastive Learning (SupCon)

Trains the MTRCNN backbone with a **supervised contrastive loss** (Khosla et al. 2020) on species labels.
A projection head (`32→64→128`, L2-normalised) is attached to the 32-dim embedding during training only.
The contrastive objective pulls same-species embeddings together across domains, targeting a lower DSG.

`supcon_weight` controls the λ scaling of the SupCon loss added to the CE losses.
`supcon_temperature` controls τ in the NT-Xent denominator (lower = sharper contrast).

**Before running:** Runtime → Change runtime type → A100 GPU (Colab Pro)

In [ ]:
# Confirm GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — check Runtime settings')

In [ ]:
# Clone repo if needed, then pull latest and install deps
import os
if not os.path.exists('/content/CD-MSC'):
    !git clone https://github.com/saha23s/CD-MSC.git /content/CD-MSC
%cd /content/CD-MSC
!git checkout aaron/preprocessing
!git pull origin aaron/preprocessing
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive (needed to save outputs at the end)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Restore pre-extracted features from Drive
# (~5 GB pkl files — too large for the repo, saved here by colab_quickstart.ipynb)
import shutil, pathlib

src = pathlib.Path('/content/drive/MyDrive/CD-MSC-feature')
dst = pathlib.Path('Development_data/feature')
dst.mkdir(parents=True, exist_ok=True)

if src.exists():
    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
    pkls = list(dst.glob('*.pkl'))
    print(f'Features restored: {len(pkls)} pkl files.')
else:
    print('ERROR: MyDrive/CD-MSC-feature not found on Drive.')
    print('Run colab_quickstart.ipynb first to extract and back up features.')

In [ ]:
# ── edit these ───────────────────────────────────────────────────────────────
SUPCON_WEIGHT   = 1.0   # λ on the contrastive loss (0.0 = CE only, try 0.5, 1.0, 2.0)
SUPCON_TEMP     = 0.1   # τ temperature (0.1 is a good starting point)
SEED            = 42
BALANCE_BATCHES = True  # True = joint species+domain balanced sampling (recommended for SupCon)
BATCH_SIZE      = 128   # larger = more in-batch negatives for contrastive loss
DANN_ALPHA_MAX  = 0.0   # 0.0 = no GRL (SupCon replaces domain adversarial objective)
CDANN           = False
SPEC_AUGMENT    = False
CMN             = False
D5_NOISE_STD    = 0.0
USE_DELTA       = False
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Write config for this experiment
import json

with open('configs/default_experiment.json') as f:
    cfg = json.load(f)

cfg['seed'] = SEED
cfg['supcon_weight'] = SUPCON_WEIGHT
cfg['supcon_temperature'] = SUPCON_TEMP
cfg['batch_balance_domain'] = BALANCE_BATCHES
cfg['batch_size'] = BATCH_SIZE
cfg['dann_alpha_max'] = DANN_ALPHA_MAX
cfg['cdann'] = CDANN
cfg['spec_augment'] = SPEC_AUGMENT
cfg['cmn'] = CMN
cfg['d5_noise_std'] = D5_NOISE_STD
cfg['use_delta'] = USE_DELTA

balanced_tag = '_balanced' if BALANCE_BATCHES else ''
supcon_tag   = f'_supcon{SUPCON_WEIGHT}'
config_path  = f'configs/supcon_w{SUPCON_WEIGHT}_t{SUPCON_TEMP}_seed{SEED}{balanced_tag}.json'
with open(config_path, 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Config written: {config_path}')
print(f'  seed             = {SEED}')
print(f'  supcon_weight    = {SUPCON_WEIGHT}')
print(f'  supcon_temp      = {SUPCON_TEMP}')
print(f'  balance_batches  = {BALANCE_BATCHES}')
print(f'  batch_size       = {BATCH_SIZE}')
print(f'  dann_alpha_max   = {DANN_ALPHA_MAX}')
print()
print('Output dir will be:')
print(f'  outputs/MTRCNN_seed{SEED}_B{BATCH_SIZE}_E100_earlystop_min20_pati10{balanced_tag}{supcon_tag}/')

## Train

Expect ~30–60 min on A100. Early stopping monitors `val species_balanced_accuracy` (min epoch 20, patience 10).

The `train_supcon_loss` in the log tracks the contrastive term separately from CE losses.
Watch for it to decrease over epochs — if it stays flat, the batch size may be too small for enough in-batch positives.

In [ ]:
!python train.py --config $config_path

In [ ]:
# Save outputs to Drive before session expires
import shutil
shutil.copytree('outputs', '/content/drive/MyDrive/CD-MSC-outputs', dirs_exist_ok=True)
print('Outputs saved to Google Drive.')

## Results

Compares the released 10-seed baseline against all DANN runs found in `outputs/`.

In [ ]:
import json, pathlib

def load_metrics(output_dir):
    p = pathlib.Path(output_dir) / 'best_model_eval' / 'test_metrics.json'
    return json.loads(p.read_text()) if p.exists() else None

rows = []

# Released 10-seed baseline mean (hard-coded from the paper)
rows.append({'run': 'Baseline 10-seed mean (released)', 'BA_seen': 0.8806, 'BA_unseen': 0.1751, 'DSG': 0.7055})

# Baseline seed 42 checkpoint (in repo)
m = load_metrics('outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5')
if m:
    rows.append({'run': 'Baseline seed 42 (repo checkpoint)', 'BA_seen': m['BA_seen'], 'BA_unseen': m['BA_unseen'], 'DSG': m['DSG']})

# All experiment runs (any subfolder with best_model_eval/test_metrics.json)
baseline_dir = 'MTRCNN_seed42_B64_E100_earlystop_min10_pati5'
for p in sorted(pathlib.Path('outputs').glob('*/best_model_eval/test_metrics.json')):
    run_name = p.parts[-3]
    if run_name == baseline_dir:
        continue  # already added above
    m = json.loads(p.read_text())
    if m.get('BA_seen') is None:
        continue
    rows.append({'run': run_name, 'BA_seen': m['BA_seen'], 'BA_unseen': m['BA_unseen'], 'DSG': m['DSG']})

if not rows:
    print('No results found yet — run training first.')
else:
    w = 70
    print(f"{'Run':<{w}} {'BA_seen':>8} {'BA_unseen':>10} {'DSG':>8}")
    print('-' * (w + 30))
    for r in rows:
        print(f"{r['run']:<{w}} {r['BA_seen']:>8.4f} {r['BA_unseen']:>10.4f} {r['DSG']:>8.4f}")